# SEG Multi-Seed Benchmark — Tox21

Kompaktes Notebook für Experimente mit verschiedenen Seeds.
Multi-label Klassifikation (12 Assays, NaN-masked BCE).

**Workflow:**
1. Zellen 1-4 einmal ausführen (Setup, Daten, Embeddings, Funktionen)  
2. Config definieren und `run_single_seed(config, seed)` aufrufen  
3. Oder: `run_multi_seed(config, seeds=[...])` für aggregierte Ergebnisse

In [1]:
# === Setup (einmal ausführen) ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch
import deepchem as dc
from typing import Dict, List, Any, Optional

print(f"Workspace: {workspace_root}")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Workspace: c:\Users\robsc\Home\Dev\molfusion2
PyTorch: 2.6.0+cu124, CUDA: True


In [2]:
# === Default Configuration ===
DEFAULT_CONFIG = {
    # Architecture
    "hidden_channels": 128,
    "K": 4,
    "num_layers": 2,
    "pool": "set2set",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",
    "fusion_dim": 64,
    "fusion_n_heads": 8,
    "text_projection_dim": 64,
    "text_proj_init": "xavier",
    "text_proj_init_gain": 0.1,
    "freeze_text_proj": True,
    
    # Regularization
    "dropout": 0.3,
    "fusion_dropout": 0.3,
    "head_dropout": 0.6,
    "weight_decay": 1e-1,
    
    # Head
    "head_type": "mlp",
    "head_hidden_dim": 64,
    
    # Training
    "learning_rate": 3e-4,
    "batch_size": 64,
    "num_epochs": 100,
    "patience": 20,
    "scheduler": "cosine",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    "grad_clip": None,
    
    # Multi-label
    "num_tasks": 12,
}

print("Default config loaded.")
print(f"Tasks: {DEFAULT_CONFIG['num_tasks']} assays")

Default config loaded.
Tasks: 12 assays


In [3]:
# === Load Tox21 Dataset (einmal ausführen) ===
import os
from deepchem.molnet.load_function.tox21_datasets import TOX21_URL, TOX21_TASKS
from rdkit import Chem

SPLIT_TYPE = "random"  # Options: "scaffold" (harder, realistic) or "random" (easier)

# Download if needed
data_dir = dc.utils.data_utils.get_data_dir()
dataset_file = os.path.join(data_dir, "tox21.csv.gz")
if not os.path.exists(dataset_file):
    dc.utils.data_utils.download_url(url=TOX21_URL, dest_dir=data_dir)

# Load CSV directly
df = pd.read_csv(dataset_file)
TOX21_TASK_LIST = TOX21_TASKS

# Extract SMILES and labels
all_smiles_raw = df['smiles'].tolist()
all_y_raw = df[TOX21_TASK_LIST].values.astype(np.float32)  # Shape: (N, 12), NaN for missing

# Filter valid SMILES
mols = [Chem.MolFromSmiles(s) for s in all_smiles_raw]
valid_idx = [i for i, m in enumerate(mols) if m is not None]
valid_smiles_all = [all_smiles_raw[i] for i in valid_idx]
valid_y_all = all_y_raw[valid_idx]

# Split
if SPLIT_TYPE == "scaffold":
    from deepchem.splits import ScaffoldSplitter
    splitter = ScaffoldSplitter()
else:
    from deepchem.splits import RandomSplitter
    splitter = RandomSplitter()

train_idx, val_idx, test_idx = splitter.split(
    dc.data.NumpyDataset(X=np.zeros((len(valid_smiles_all), 1)), y=valid_y_all, ids=valid_smiles_all),
    frac_train=0.8, frac_valid=0.1, frac_test=0.1,
    **({"seed": 42} if SPLIT_TYPE == "random" else {})
)

TRAIN_SMILES = [valid_smiles_all[i] for i in train_idx]
TRAIN_Y = valid_y_all[train_idx]
VALID_SMILES = [valid_smiles_all[i] for i in val_idx]
VALID_Y = valid_y_all[val_idx]
TEST_SMILES = [valid_smiles_all[i] for i in test_idx]
TEST_Y = valid_y_all[test_idx]

print(f"Tasks ({len(TOX21_TASK_LIST)}): {TOX21_TASK_LIST}")
print(f"Split: {SPLIT_TYPE} | Train: {len(TRAIN_SMILES)} | Valid: {len(VALID_SMILES)} | Test: {len(TEST_SMILES)}")
print(f"Train labels shape: {TRAIN_Y.shape}")

[00:52:11] WARNING: not removing hydrogen atom without neighbors
[00:52:11] Explicit valence for atom # 8 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 3 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 4 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 9 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 5 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 16 Al, 6, is greater than permitted
[00:52:11] Explicit valence for atom # 20 Al, 6, is greater than permitted


Tasks (12): ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']
Split: random | Train: 6258 | Valid: 782 | Test: 783
Train labels shape: (6258, 12)


In [4]:
# === Load Text Embeddings (einmal ausführen) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_EMB_DIR = workspace_root / "cache" / "cot_embeddings"
TASK = "toxicity_fast"

npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = TRAIN_SMILES + VALID_SMILES + TEST_SMILES
all_emb = torch.from_numpy(cache.get_batch(all_smiles))

n_train, n_valid = len(TRAIN_SMILES), len(VALID_SMILES)
TRAIN_TEXT_EMB = all_emb[:n_train]
VALID_TEXT_EMB = all_emb[n_train:n_train + n_valid]
TEST_TEXT_EMB = all_emb[n_train + n_valid:]

print(f"\u2713 Loaded {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={TRAIN_TEXT_EMB.shape}, valid={VALID_TEXT_EMB.shape}, test={TEST_TEXT_EMB.shape}")

Loading embeddings from toxicity_fast_text_embeddings_compact.npz...
  Loaded 7823 entries, dim=3072
  Memory mode: mapped
✓ Loaded toxicity_fast_text_embeddings_compact.npz (7823 molecules)
  Embeddings: train=torch.Size([6258, 3072]), valid=torch.Size([782, 3072]), test=torch.Size([783, 3072])


In [5]:
# === Experiment Functions (einmal ausführen) ===
from itertools import product
from sklearn.metrics import roc_auc_score
from models import SEGPredictor, SEGPredictorConfig


def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def multilabel_auc(y_true, y_pred, tasks):
    """Per-task AUC-ROC with NaN masking, returns mean AUC and per-task AUCs."""
    aucs = []
    for i in range(len(tasks)):
        valid_mask = ~np.isnan(y_true[:, i])
        yt = y_true[valid_mask, i]
        yp = y_pred[valid_mask, i]
        if len(np.unique(yt)) < 2 or len(yt) < 10:
            aucs.append(np.nan)
            continue
        aucs.append(float(roc_auc_score(yt, yp)))
    valid_aucs = [a for a in aucs if not np.isnan(a)]
    mean_auc = float(np.mean(valid_aucs)) if valid_aucs else np.nan
    return {"mean_auc": mean_auc, "per_task_aucs": aucs, "n_valid_tasks": len(valid_aucs)}


def run_single_seed(
    config: Dict[str, Any],
    seed: int,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Run a single SEG experiment with the given config and seed.
    """
    cfg = {**DEFAULT_CONFIG, **config}
    set_seed(seed)
    
    seg_config = SEGPredictorConfig(
        task="classification",
        num_tasks=cfg["num_tasks"],
        hidden_channels=cfg["hidden_channels"],
        K=cfg["K"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        pool=cfg["pool"],
        set2set_processing_steps=cfg["set2set_processing_steps"],
        text_embedding_dim=3072,
        text_projection_dim=cfg["fusion_dim"],
        text_proj_init=cfg["text_proj_init"],
        text_proj_init_gain=cfg["text_proj_init_gain"],
        freeze_text_proj=cfg["freeze_text_proj"],
        fusion=cfg["fusion"],
        fusion_dim=cfg["fusion_dim"],
        fusion_n_heads=cfg.get("fusion_n_heads", 8),
        fusion_dropout=cfg["fusion_dropout"],
        head_type=cfg["head_type"],
        head_hidden_dim=cfg["head_hidden_dim"],
        head_dropout=cfg["head_dropout"],
    )
    
    seg = SEGPredictor(config=seg_config)
    
    if verbose:
        print(f"=== Seed {seed} ===")
    
    history = seg.fit(
        smiles_list=TRAIN_SMILES,
        labels=TRAIN_Y,
        val_smiles=VALID_SMILES,
        val_labels=VALID_Y,
        text_embeddings=TRAIN_TEXT_EMB,
        val_text_embeddings=VALID_TEXT_EMB,
        num_epochs=cfg["num_epochs"],
        batch_size=cfg["batch_size"],
        learning_rate=cfg["learning_rate"],
        weight_decay=cfg["weight_decay"],
        patience=cfg["patience"],
        scheduler=cfg["scheduler"],
        scheduler_patience=cfg["scheduler_patience"],
        scheduler_factor=cfg["scheduler_factor"],
        min_lr=cfg["min_lr"],
        grad_clip=cfg["grad_clip"],
        seed=seed,
        verbose=verbose,
    )
    
    preds = seg.predict_batch(TEST_SMILES, text_embeddings=TEST_TEXT_EMB)
    metrics = multilabel_auc(TEST_Y, preds, TOX21_TASK_LIST)
    
    if verbose:
        print(f"\u2192 Test Mean AUC={metrics['mean_auc']:.4f} ({metrics['n_valid_tasks']} tasks)\n")
    
    return {"seed": seed, "metrics": metrics, "history": history, "model": seg}


def run_multi_seed(
    config: Dict[str, Any],
    seeds: List[int],
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Run experiments with multiple seeds and aggregate results.
    """
    results = []
    
    print(f"Running {len(seeds)} experiments with seeds: {seeds}")
    print("-" * 60)
    
    for i, seed in enumerate(seeds):
        print(f"[{i+1}/{len(seeds)}] Seed {seed}...", end=" ", flush=True)
        result = run_single_seed(config, seed, verbose=verbose)
        results.append(result)
        if not verbose:
            m = result["metrics"]
            print(f"Mean AUC={m['mean_auc']:.4f} ({m['n_valid_tasks']} tasks)")
    
    df = pd.DataFrame([
        {"seed": r["seed"], "mean_auc": r["metrics"]["mean_auc"], "n_valid_tasks": r["metrics"]["n_valid_tasks"]}
        for r in results
    ])
    
    summary = {
        "mean_auc_mean": df["mean_auc"].mean(),
        "mean_auc_std": df["mean_auc"].std(),
        "n_runs": len(seeds),
        "seeds": seeds,
    }
    
    print("-" * 60)
    print(f"\n=== AGGREGATED RESULTS ({len(seeds)} seeds) ===")
    print(f"Mean AUC-ROC: {summary['mean_auc_mean']:.4f} \u00b1 {summary['mean_auc_std']:.4f}")
    
    return {"summary": summary, "runs": results, "df": df}


def run_grid_search(
    grid_config: Dict[str, Any],
    seeds: List[int] = [42],
    sort_by: str = "mean_auc",
    ascending: bool = False,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Grid search over hyperparameters with multi-seed evaluation.
    
    Values that are lists \u2192 swept over (all combinations).
    Values that are scalars \u2192 fixed for all runs.
    """
    sweep_keys = []
    sweep_values = []
    fixed_params = {}
    
    for k, v in grid_config.items():
        if isinstance(v, list):
            sweep_keys.append(k)
            sweep_values.append(v)
        else:
            fixed_params[k] = v
    
    if sweep_keys:
        combos = list(product(*sweep_values))
    else:
        combos = [()]
    
    n_combos = len(combos)
    n_total = n_combos * len(seeds)
    
    print(f"=== GRID SEARCH ===")
    if sweep_keys:
        print(f"Sweep params: {', '.join(f'{k} ({len(v)} values)' for k, v in zip(sweep_keys, sweep_values))}")
    else:
        print("No sweep params (single config)")
    print(f"Combinations: {n_combos} \u00d7 {len(seeds)} seeds = {n_total} total runs")
    print("=" * 70)
    
    all_rows = []
    run_counter = 0
    
    for combo_idx, combo in enumerate(combos):
        config = {**fixed_params}
        for k, v in zip(sweep_keys, combo):
            config[k] = v
        
        combo_desc = ", ".join(f"{k}={v}" for k, v in zip(sweep_keys, combo)) if sweep_keys else "default"
        print(f"\n[{combo_idx+1}/{n_combos}] {combo_desc}")
        
        seed_metrics = []
        for seed in seeds:
            run_counter += 1
            print(f"  ({run_counter}/{n_total}) seed={seed}...", end=" ", flush=True)
            result = run_single_seed(config, seed, verbose=verbose)
            seed_metrics.append(result["metrics"])
            m = result["metrics"]
            print(f"Mean AUC={m['mean_auc']:.4f}")
        
        aucs = [m["mean_auc"] for m in seed_metrics]
        
        row = {**{k: v for k, v in zip(sweep_keys, combo)}}
        row["mean_auc_mean"] = np.mean(aucs)
        row["mean_auc_std"] = np.std(aucs)
        row["n_seeds"] = len(seeds)
        all_rows.append(row)
    
    df = pd.DataFrame(all_rows)
    
    sort_col = f"{sort_by}_mean"
    if sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=ascending).reset_index(drop=True)
    
    print("\n" + "=" * 70)
    print(f"GRID SEARCH RESULTS (sorted by {sort_by})")
    print("=" * 70)
    
    for i, row in df.iterrows():
        params = " | ".join(f"{k}={row[k]}" for k in sweep_keys) if sweep_keys else "default"
        print(f"  #{i+1}: {params}")
        print(f"      Mean AUC={row['mean_auc_mean']:.4f}\u00b1{row['mean_auc_std']:.4f}")
    
    return df


print("\u2713 Functions loaded: run_single_seed, run_multi_seed, run_grid_search")

✓ Functions loaded: run_single_seed, run_multi_seed, run_grid_search


---
## Experimente

Ab hier: Config definieren und Experimente starten.

In [6]:
# === Einzelnes Experiment ===
# Config-Overrides (leer = Default-Config verwenden)
my_config = {
    # Hier eigene Werte überschreiben, z.B.:
    # "dropout": 0.5,
    # "weight_decay": 5e-2,
}

#result = run_single_seed(my_config, seed=42)

In [7]:
# === Multi-Seed Experiment ===
my_config = {}  # Default-Config

#results = run_multi_seed(my_config, seeds=[42, 123, 456, 789, 1337])

In [8]:
# === Ergebnisse anzeigen ===
#results["df"]

---
## Varianten testen

Config anpassen und erneut ausführen:

In [11]:
# === Grid Search Beispiel ===
# Werte als Liste → werden gesweept (alle Kombinationen)
# Werte als Skalar → bleiben fix
grid_config = {
    "hidden_channels" : [64, 128],
    "K" : [3, 4],
    "text_proj_init": ["xavier"],
    #"pool": ["sum", "set2set"],
    #"freeze_text_proj": [True, False],
    #"fusion_dim": [32, 64],
    "fusion": "cross_mha",       # fix
    "head_type": "mlp",          # fix
}

# Grid search mit 4 Seeds pro Kombination
grid_df = run_grid_search(grid_config, seeds=[42, 113])

=== GRID SEARCH ===
Sweep params: hidden_channels (2 values), K (2 values), text_proj_init (1 values)
Combinations: 4 × 2 seeds = 8 total runs

[1/4] hidden_channels=64, K=3, text_proj_init=xavier
  (1/8) seed=42... 

[00:53:08] WARNING: not removing hydrogen atom without neighbors


Mean AUC=0.8125
  (2/8) seed=113... 

[00:56:23] WARNING: not removing hydrogen atom without neighbors


Mean AUC=0.8044

[2/4] hidden_channels=64, K=4, text_proj_init=xavier
  (3/8) seed=42... 

...

Mean AUC=0.8069

[4/4] hidden_channels=128, K=4, text_proj_init=xavier
  (7/8) seed=42... 

[01:11:07] WARNING: not removing hydrogen atom without neighbors


Mean AUC=0.8149
  (8/8) seed=113... 

[01:13:55] WARNING: not removing hydrogen atom without neighbors


Mean AUC=0.8133

GRID SEARCH RESULTS (sorted by mean_auc)
  #1: hidden_channels=128 | K=4 | text_proj_init=xavier
      Mean AUC=0.8141±0.0008
  #2: hidden_channels=128 | K=3 | text_proj_init=xavier
      Mean AUC=0.8140±0.0071
  #3: hidden_channels=64 | K=4 | text_proj_init=xavier
      Mean AUC=0.8117±0.0029
  #4: hidden_channels=64 | K=3 | text_proj_init=xavier
      Mean AUC=0.8085±0.0040


In [12]:
# === Grid Search Ergebnisse ===
grid_df

,hidden_channels,K,text_proj_init,mean_auc_mean,mean_auc_std,n_seeds
0,128,4,xavier,0.814081,0.000799,2
1,128,3,xavier,0.814009,0.007079,2
2,64,4,xavier,0.811725,0.002889,2
3,64,3,xavier,0.808475,0.004028,2


In [ ]:
# === Grid Search Ergebnisse speichern ===
out_path = workspace_root / "benchmarking" / "results" / "grid_search_tox21_2.csv"
grid_df.to_csv(out_path, index=False)
print(f"\u2713 Saved to {out_path}")

✓ Saved to c:\Users\robsc\Home\Dev\molfusion2\benchmarking\results\grid_search_tox21_1.csv
